<a href="https://colab.research.google.com/github/noa-bedoya/boltz-colab/blob/main/BOLTZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install boltz

In [ ]:
!rm -rf ~/.boltz/mols.tar ~/.boltz/mols

In [ ]:
!rm -rf ~/.boltz/

In [ ]:
#@markdown ### Files configuration to run Boltz
#@markdown Specify the project name:
project_name = "prot_xxxx.yaml" #@param {type:"string"}

!boltz predict {project_name} --use_msa_server --out_dir results_boltz


In [ ]:
#@title Files configuration to visualize the 3D structure {run: "auto"}

import py3Dmol
import glob

#@markdown ### File configuration
#@markdown Ensure the names match your previous command:
results_directory = "./results/boltz_results_prot_xxxx/predictions" #@param {type:"string"}
project_name = "prot_xxxx" #@param {type:"string"}

#@markdown ### Visualization options
color = "pLDDT (Confidence)" #@param ["pLDDT (Confidence)", "rainbow", "chain"]
show_side_chains = False #@param {type:"boolean"}

def visualize_boltz():
    # Boltz usually saves the final file in .cif format, searching in the folder
    search_path = f"{results_directory}/{project_name}/*_model_*.cif"
    files = glob.glob(search_path)

    # If no .cif is found, search for a .pdb just in case
    if not files:
        files = glob.glob(f"{results_directory}/{project_name}/*.pdb") + glob.glob(f"{results_directory}/{project_name}/*.cif")

    if not files:
        print(f"❌ No structure file found in: {results_directory}/{project_name}/")
        print("Make sure the prediction finished successfully.")
        return

    final_file = files[0]
    print(f"Displaying structure: {final_file}")

    # Initialize the 3D viewer
    view = py3Dmol.view(width=800, height=500)
    file_format = 'cif' if final_file.endswith('.cif') else 'pdb'

    with open(final_file, 'r') as f:
        view.addModel(f.read(), file_format)

    # Configure the color scheme
    if color == "pLDDT (Confidence)":
        # Uses the classic scale (red=bad, blue=excellent) based on the B-factor
        view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
    elif color == "rainbow":
        view.setStyle({'cartoon': {'color':'spectrum'}})
    elif color == "chain":
        view.setStyle({'cartoon': {'colorscheme':'chain'}})

    # Draw side chains (the amino acid "sticks")
    if show_side_chains:
        main_atoms = ['C','O','N']
        view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':main_atoms,'invert':True}]},
                      {'stick':{'colorscheme':"WhiteCarbon",'radius':0.2}})

    view.zoomTo()
    view.show()

visualize_boltz()

In [ ]:
#@title Boltz Plots (PAE, PDE) {run: "auto"}

import numpy as np
import matplotlib.pyplot as plt

#@markdown ### File configuration
#@markdown Ensure the names match your previous command:
results_directory = "./results/boltz_results_prot_xxxx/predictions" #@param {type:"string"}
project_name = "prot_xxxx" #@param {type:"string"}



pae_data = np.load(f"{results_directory}/{project_name}/pae_{project_name}_model_0.npz")
pae_matrix = pae_data[pae_data.files[0]]

pde_data = np.load(f"{results_directory}/{project_name}/pde_{project_name}_model_0.npz")
pde_matrix = pde_data[pde_data.files[0]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# PAE Plot
im1 = ax1.imshow(pae_matrix, cmap='Greens_r', vmin=0, vmax=30) # Dark green = lower error
ax1.set_title('PAE (Predicted Aligned Error)')
ax1.set_xlabel('Residue')
ax1.set_ylabel('Residue')
fig.colorbar(im1, ax=ax1, label='Error (Å)')

# PDE Plot
im2 = ax2.imshow(pde_matrix, cmap='magma') # Magma is excellent for spotting diffusion error hotspots
ax2.set_title('PDE (Predicted Diffusion Error)')
ax2.set_xlabel('Residue')
ax2.set_ylabel('Residue')
fig.colorbar(im2, ax=ax2, label='Diffusion Error')

plt.tight_layout()
plt.show()

In [ ]:
#@title Results Download {run: "auto"}

import os
from google.colab import files

#@markdown ### Download configuration
#@markdown Paste the exact folder path here:
exact_folder_path = "/content/results/boltz_results_prot_xxxx" #@param {type:"string"}
final_zip_name = "my_results_xxxx.zip" #@param {type:"string"}

if os.path.exists(exact_folder_path):
    print("Folder found! Compressing...")
    # Compress the folder
    !zip -q -r {final_zip_name} {exact_folder_path}

    print("Downloading...")
    # Download the file
    files.download(final_zip_name)
else:
    print(f"❌ ERROR: Folder not found at path: {exact_folder_path}")
    print("Please make sure you have copied the path correctly from the left sidebar.")